# CASMI 2026 — candidate retrieval + pretrained FPNet + NP-aware blend (v5)

Extends the library-retrieval baseline to **classes 2/3-ish** (structure known to PubChem/COCONUT, no public spectra):

1. **Candidate pool** = `pool_popularity.csv` (710k training + COCONUT structures with PubChem popularity) ∪ natural-product knowledge table (LOTUS/NPAtlas/COCONUT, 152k).
2. Per test molecule: shortlist candidates by neutral mass.
3. Score each candidate with
   * `lib` — best-match spectral cosine against library spectra (0 if the structure has no library spectra),
   * `fp` — expected Tanimoto between the fingerprint predicted from the molecule's spectra (public **FPNet** transformer: `ahmedberatozer/casmi26-v3-models` for the test, the fold-safe checkpoint `dmitriigluzdov/casmi26-fold-safe-fpnet` for honest validation) and the candidate's Morgan fingerprint, z-scored within the shortlist,
   * `pop` — PubChem popularity prior (`log1p(substances)+log1p(pubmed)`),
   * `np` — natural-product flag.
4. (v5) **Natural-product retuning**: the blend is tuned on three validation groups — class-1-like, general held-out structures, and the 265 held-out *natural-product panel* structures (same timsTOF pipeline as the test) — with extra features `np_narrow` (LOTUS/NPAtlas membership). Blend with weights tuned on a local validation mix (class-1-like and held-out-structure class-2-like molecules), output top 25.

Data used: competition data, `dmitriigluzdov/casmi26-pubchem-popularity-prior`, `dmitriigluzdov/casmi26-natural-product-knowledge-table`, `ahmedberatozer/casmi26-v2-pool` (precomputed candidate fingerprints) and the public `casmi` code in `casmi26-v3-models`.

In [ ]:
import sys, subprocess, importlib.metadata, glob
try: _v = importlib.metadata.version("rdkit")
except importlib.metadata.PackageNotFoundError: _v = ""
if not _v:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    w = sorted(glob.glob(f"/kaggle/input/**/rdkit-*-{tag}-*.whl", recursive=True))
    assert w, f"attach an offline RDKit wheel for {tag}"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q", w[-1]])


In [ ]:
import os, re, sys, time, glob, itertools
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
import scipy.sparse as sp, torch, torch.nn as nn
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")
def find(name, env):
    p = os.environ.get(env) or next(iter(glob.glob(f"/kaggle/input/**/{name}", recursive=True)), None)
    assert p, f"{name} not found"; return p
TRAIN = find("train.parquet", "CASMI_TRAIN"); DATA_DIR = os.path.dirname(TRAIN)
POOL_CSV, NP_PQ = find("pool_popularity.csv", "POOL_CSV"), find("natural_products.parquet", "NP_PQ")
CODE_DIR = os.path.dirname(os.path.dirname(find("fpnet.py", "CASMI_FPNET_PY")))   # .../code (contains casmi/)
sys.path.insert(0, CODE_DIR)
CKPT_PUBLIC = [find("fpnet_0.pt", "CKPT0"), find("fpnet_1.pt", "CKPT1")]            # trained on (almost) all training molecules
CKPT_FOLD0 = find("fpnet_fold0.pt", "CKPT_FOLD0")                                    # honest: never saw the held-out structures
SPLIT_PQ = find("split.parquet", "SPLIT_PQ")
POOL_FP = find("pool_fp.npy", "POOL_FP"); FP_BITS_NPY = find("fp_bits.npy", "FP_BITS")
OUT_DIR = os.environ.get("CASMI_OUT", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
CFG = dict(bin_width=0.02, max_mz=1300.0, min_rel_int=0.01, max_peaks=100, min_peaks=2,
           mass_tol_da=0.01, mass_tol_ppm=10.0, max_cands=4000, topk=25)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(TRAIN, POOL_CSV, NP_PQ, CKPT_PUBLIC, CKPT_FOLD0, dev)

In [ ]:
# ---- masses ----
EL = {"H":1.00782503207,"C":12.0,"N":14.0030740048,"O":15.99491461956,"F":18.99840322,"Na":22.9897692809,
      "P":30.97376163,"S":31.972071,"Cl":34.96885268,"K":38.96370668,"Br":78.9183371,"I":126.904473,
      "B":11.0093054,"Si":27.9769265325,"Se":79.9165213,"As":74.9215965,"Fe":55.9349375,"Mg":23.985041699,
      "Ca":39.96259098,"Zn":63.9291422,"Cu":62.9295975,"Li":7.016003437,"Pt":194.964791,"Sn":119.9022016,
      "Hg":201.970643,"Co":58.933195,"Mn":54.9380451,"Ni":57.9353429,"Al":26.98153863,"Ge":73.9211778,"Sb":120.903818}
_tok = re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str) or not f: return np.nan
    m = 0.0
    for el, n in _tok.findall(f):
        if el not in EL: return np.nan
        m += EL[el] * (int(n) if n else 1)
    return m

PROTON, E = 1.007276467, 0.000548580
H2O, NH4, NA, K, CL, CH2O2 = 18.0105646863, 18.0343740, 22.9897692809, 38.96370668, 34.96885268, 46.00547930326
# neutral M = precursor_mz - ADDUCT_DELTA
ADDUCT_DELTA = {
    "[M+H]+": PROTON, "[M+NH4]+": NH4 + 0 - E, "[M-H2O+H]+": PROTON - H2O, "[M-2H2O+H]+": PROTON - 2*H2O,
    "[M+Na]+": NA - E, "[M+K]+": K - E, "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON - H2O,
    "[M+CH2O2-H]-": CH2O2 - PROTON, "[M+Cl]-": CL + E,
}
def neutral_mass(df):
    d = df["adduct"].map(ADDUCT_DELTA)
    return (df["precursor_mz"] - d).to_numpy(float)

def mol_masses(test_df):
    d = test_df.assign(nm=neutral_mass(test_df))
    return d.groupby("molecule_id")["nm"].median()

In [ ]:
# ---- spectrum -> sparse vector ----
NB = int(CFG["max_mz"] / CFG["bin_width"]) + 1
def vectorize(mzs_list, ints_list, prec):
    rows, cols, vals = [], [], []
    keep = np.zeros(len(mzs_list), bool)
    for i, (mz, it, p) in enumerate(zip(mzs_list, ints_list, prec)):
        mz = np.asarray(mz, float); it = np.asarray(it, float)
        if len(mz) == 0: continue
        it = it / it.max()
        m = (mz <= p + 2.0) & (it >= CFG["min_rel_int"]) & (mz < CFG["max_mz"]) & (mz > 0)
        mz, it = mz[m], it[m]
        if len(mz) < CFG["min_peaks"]: continue
        if len(mz) > CFG["max_peaks"]:
            top = np.argsort(-it)[:CFG["max_peaks"]]; mz, it = mz[top], it[top]
        b = (mz / CFG["bin_width"]).astype(np.int64)
        v = np.sqrt(it)
        s = pd.Series(v).groupby(b).max()          # merge peaks sharing a bin
        v = s.to_numpy(); b = s.index.to_numpy()
        v = v / np.linalg.norm(v)
        rows.append(np.full(len(b), i)); cols.append(b); vals.append(v); keep[i] = True
    if not rows: return sp.csr_matrix((len(mzs_list), NB)), keep
    X = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(len(mzs_list), NB))
    return X, keep

In [ ]:
# ---- candidate pool ----
pool = pd.read_csv(POOL_CSV, usecols=["pool_row","inchikey14","smiles","monoisotopic_mass","pop","np_coconut","np_lotus","np_npatlas"])
pool["np"] = ((pool.np_coconut + pool.np_lotus + pool.np_npatlas) > 0).astype(np.float32)
pool["nn"] = ((pool.np_lotus + pool.np_npatlas) > 0).astype(np.float32)
pool = pool.rename(columns={"monoisotopic_mass": "mass", "pool_row": "pid"})[["pid","inchikey14","smiles","mass","pop","np","nn"]]
npt = pd.read_parquet(NP_PQ, columns=["ik14","smiles","mono_mass","pubchem_substances","pubmed_articles","in_lotus","in_npatlas"])
npt = npt[~npt.ik14.isin(set(pool.inchikey14))]
npt = pd.DataFrame({"pid": -1, "inchikey14": npt.ik14, "smiles": npt.smiles, "mass": npt.mono_mass,
                    "pop": np.log1p(npt.pubchem_substances) + np.log1p(npt.pubmed_articles), "np": 1.0, "nn": (npt.in_lotus | npt.in_npatlas).astype(np.float32).to_numpy()})
pool = pd.concat([pool, npt], ignore_index=True).dropna(subset=["mass","smiles"]).sort_values("mass").reset_index(drop=True)
pool_mass = pool.mass.to_numpy(); pool_key = pool.inchikey14.to_numpy()
print(len(pool), "candidate structures;", int(pool.np.sum()), "natural products")
lib_keys = set(pq.read_table(TRAIN, columns=["inchikey14"]).column(0).unique().to_pylist())
print(len(lib_keys), "structures with library spectra")

In [ ]:
# ---- pretrained FPNet + candidate fingerprints ----
import torch
from casmi import fpnet, chem
bits = np.load(FP_BITS_NPY); NBITS = len(bits)
pool_fp = np.load(POOL_FP, mmap_mode="r")
pool_pid = pool.pid.to_numpy(); pool_smi_all = pool.smiles.to_numpy()
bank_val = fpnet.ModelBank([CKPT_FOLD0], dev)
bank_test = fpnet.ModelBank(CKPT_PUBLIC, dev)
print("FPNet bits", NBITS, dev)
_extra_fp = {}
def cand_fp_packed(i):
    """packed fingerprint for pool row i (precomputed if in the public pool, else computed)."""
    if pool_pid[i] >= 0: return np.asarray(pool_fp[pool_pid[i]])
    k = pool_key[i]
    if k not in _extra_fp:
        r = chem.raw_fingerprint(pool_smi_all[i]); _extra_fp[k] = np.packbits(r[bits]) if r is not None else np.zeros((NBITS + 7) // 8, np.uint8)
    return _extra_fp[k]

def mol_z(g, bank):
    """Mean per-spectrum fingerprint logits (nbits,) for one molecule."""
    items = []
    for r in g.itertuples():
        pm, pi = fpnet.prep_peaks(r.ms2_mzs, r.ms2_normalized_intensities, r.precursor_mz)
        ce = getattr(r, "collision_energy_ev", None)
        ce = float(np.mean(ce)) if ce is not None and len(ce) else None
        items.append(dict(mz=pm, it=pi, prec=float(r.precursor_mz), adduct_ix=chem.adduct_index(r.adduct), ce=ce or 0.0,
                          ce_known=1.0 if ce else 0.0, n_merged=1, mode=1.0 if r.ionization_mode == "positive" else -1.0))
    return bank.logits(items).mean(0)

def fp_scores(idx, z):
    P = torch.from_numpy(np.stack([cand_fp_packed(i) for i in idx])).to(dev)
    F = fpnet.unpack_gpu(P, NBITS)
    return (F @ torch.from_numpy(z.astype(np.float32)).to(dev)).cpu().numpy()


In [ ]:
def load_library_vectors(keys, exclude_rows=frozenset(), exclude_keys=frozenset()):
    keys = pa.array(sorted(set(keys) - set(exclude_keys)))
    pf = pq.ParquetFile(TRAIN); cols = ["inchikey14","ms2_mzs","ms2_normalized_intensities","precursor_mz"]
    mats, metas, off = [], [], 0
    for b in pf.iter_batches(batch_size=50000, columns=cols):
        t = pa.Table.from_batches([b]); t = t.append_column("spectrum_id", pa.array(np.arange(off, off + t.num_rows))); off += t.num_rows
        t = t.filter(pc.is_in(t["inchikey14"], value_set=keys))
        if t.num_rows == 0: continue
        df = t.to_pandas()
        if exclude_rows: df = df[~df.spectrum_id.isin(exclude_rows)]
        X, keep = vectorize(df.ms2_mzs, df.ms2_normalized_intensities, df.precursor_mz.to_numpy(float))
        mats.append(X[np.flatnonzero(keep)]); metas.append(df.loc[keep, ["spectrum_id","inchikey14"]])
    X = sp.vstack(mats).tocsr() if mats else sp.csr_matrix((0, NB))
    meta = pd.concat(metas, ignore_index=True) if metas else pd.DataFrame(columns=["spectrum_id","inchikey14"])
    return X, meta

def shortlist(mass):
    tol = max(CFG["mass_tol_da"], mass * CFG["mass_tol_ppm"] * 1e-6)
    lo, hi = np.searchsorted(pool_mass, [mass - tol, mass + tol])
    idx = np.arange(lo, hi)
    if len(idx) > CFG["max_cands"]: idx = idx[np.argsort(-pool["pop"].to_numpy()[idx])[:CFG["max_cands"]]]
    return idx

def components(test_df, bank, exclude_rows=frozenset(), exclude_keys=frozenset()):
    """Per molecule: candidate idx + lib / fp / pop / np component arrays."""
    mm = mol_masses(test_df)
    cand = {m: shortlist(x) for m, x in mm.items() if np.isfinite(x)}
    allkeys = {pool_key[i] for v in cand.values() for i in v} & lib_keys
    t0 = time.time(); LX, Lmeta = load_library_vectors(allkeys, exclude_rows, exclude_keys)
    print(f"library: {LX.shape[0]} spectra / {len(allkeys)} structures ({time.time()-t0:.0f}s)")
    by_key = Lmeta.groupby("inchikey14").indices; out = {}
    for n, (mol, g) in enumerate(test_df.groupby("molecule_id")):
        idx = cand.get(mol)
        if idx is None or len(idx) == 0: out[mol] = None; continue
        keys = pool_key[idx]
        fp = fp_scores(idx, mol_z(g, bank))
        lib = np.zeros(len(idx), np.float32)
        have = [j for j, k in enumerate(keys) if k in by_key]
        if have:
            QX, keep = vectorize(g.ms2_mzs, g.ms2_normalized_intensities, g.precursor_mz.to_numpy(float)); QX = QX[np.flatnonzero(keep)]
            if QX.shape[0]:
                rows = np.concatenate([by_key[keys[j]] for j in have]); own = np.concatenate([[q]*len(by_key[keys[j]]) for q, j in enumerate(have)])
                S = (QX @ LX[rows].T).toarray()
                for q, j in enumerate(have): lib[j] = S[:, own == q].max(axis=1).mean()
        out[mol] = dict(idx=idx, lib=lib, fp=fp.astype(np.float32), pop=pool["pop"].to_numpy()[idx], np=pool["np"].to_numpy()[idx], nn=pool["nn"].to_numpy()[idx])
    return out

def blend(c, w):
    fz = (c["fp"] - c["fp"].mean()) / (c["fp"].std() + 1e-6)
    pz = (c["pop"] - c["pop"].mean()) / (c["pop"].std() + 1e-6)
    t = w.get("t", 0.0); le = np.maximum(c["lib"] - t, 0) / (1 - t)      # only strong library matches count
    return w["lib"] * le + w["fp"] * fz + w["pop"] * pz + w["np"] * c["np"] + w.get("nn", 0) * c["nn"]

def rank_keys(comps, w, k=25):
    out = {}
    for mol, c in comps.items():
        if c is None: out[mol] = []; continue
        o = np.argsort(-blend(c, w))[:k]; out[mol] = [pool_key[c["idx"][j]] for j in o]
    return out

def mrr(ranked, truth, k=25):
    rr = [(1.0 / (ranked[m].index(t) + 1)) if t in ranked.get(m, [])[:k] else 0.0 for m, t in truth.items()]
    return float(np.mean(rr)) if rr else float("nan")

## Validation mix

* **class-2 proxy**: held-out structures from the FP-model's validation split; *all* their spectra are removed from the library (the model never saw them either). Candidate pool still contains them.
* **class-1 proxy**: random other structures, half of their spectra held out and removed from the library.

Weights are tuned on a mix (30% class-1 / 70% class-2, matching the ~0.14 public score of library-only retrieval).

In [ ]:
N_EACH = int(os.environ.get("N_VAL", 200))
meta = pq.read_table(TRAIN, columns=["inchikey14","adduct"]).to_pandas(); meta["row"] = np.arange(len(meta))
meta = meta[meta.adduct.isin(ADDUCT_DELTA)]
rng = np.random.RandomState(1)
split_df = pd.read_parquet(SPLIT_PQ); vk = split_df.inchikey14[split_df.fold == "hold"].reset_index(drop=True)
ok_keys = set(meta.inchikey14) & set(pool_key)
npk = split_df.inchikey14[split_df.fold == "np"]; vnp = npk[npk.isin(ok_keys)].unique()
v2 = rng.choice(vk[vk.isin(ok_keys) & ~vk.isin(set(vnp))].to_numpy(), N_EACH, replace=False)
print(len(vnp), "natural-product panel structures usable;", len(v2), "general held-out;")
cnt = meta[~meta.inchikey14.isin(set(vk))].groupby("inchikey14").size(); m1 = cnt[cnt >= 2].index.to_numpy()
v1 = rng.choice(m1, N_EACH, replace=False)
rows2 = meta[meta.inchikey14.isin(set(v2) | set(vnp))]
rows1 = meta[meta.inchikey14.isin(set(v1))].groupby("inchikey14", group_keys=False).apply(lambda g: g.sample(frac=0.5, random_state=0) if len(g) > 2 else g.iloc[:1])
use = pd.concat([rows2, rows1]); want = set(use.row)
parts, off = [], 0; sorted_want = np.array(sorted(want))
for b in pq.ParquetFile(TRAIN).iter_batches(batch_size=50000, columns=["inchikey14","ms2_mzs","ms2_normalized_intensities","adduct","ionization_mode","precursor_mz","collision_energy_ev"]):
    lo, hi = np.searchsorted(sorted_want, [off, off + b.num_rows])
    if hi > lo:
        d = pa.Table.from_batches([b]).take(pa.array(sorted_want[lo:hi] - off)).to_pandas(); d["row"] = sorted_want[lo:hi]; parts.append(d)
    off += b.num_rows
val_df = pd.concat(parts, ignore_index=True); val_df["molecule_id"] = val_df.inchikey14
truth = {k: k for k in val_df.inchikey14.unique()}
cls = {k: (3 if k in set(vnp) else 2 if k in set(v2) else 1) for k in truth}
t0 = time.time()
vcomp = components(val_df, bank_val, exclude_rows=set(rows1.row), exclude_keys=set(v2) | set(vnp))
print(f"components done {time.time()-t0:.0f}s")

In [ ]:
GROUPS = {1: "class1", 2: "general", 3: "NP panel"}
def score_groups(w):
    r = rank_keys(vcomp, w)
    return {g: mrr(r, {m: t for m, t in truth.items() if cls[m] == g}) for g in GROUPS}
MIXW = {1: 0.15, 2: 0.25, 3: 0.60}      # test is natural-product heavy: weight the NP panel most
def score_mix(w):
    sg = score_groups(w); return sum(MIXW[g] * sg[g] for g in GROUPS), sg
def fmt(sc): return "mix %.4f | " % sc[0] + " ".join(f"{GROUPS[g]} {v:.3f}" for g, v in sc[1].items())
best = None; rows = []
for t, wl, wp, wn, wnn in itertools.product([0.5, 0.7, 0.85], [0, 2, 5, 10], [0, 0.15, 0.25, 0.5, 0.75], [0, 0.5, 1], [0, 0.5, 1, 2]):
    w = dict(t=t, lib=wl, fp=1.0, pop=wp, np=wn, nn=wnn); s = score_mix(w); rows.append((s[0], w))
    if best is None or s[0] > best[0][0]: best = (s, w)
print("BEST", fmt(best[0]), best[1])
for name, w in [("fp only", dict(lib=0, fp=1, pop=0, np=0, nn=0)), ("pop only", dict(lib=0, fp=0, pop=1, np=0, nn=0))]:
    print(name, fmt(score_mix(w)))
for sc_, w in sorted(rows, key=lambda x: -x[0])[:5]: print(" top5: %.4f" % sc_, w)
W = best[1]

## Predict test and write `submission.csv`

In [ ]:
test = pq.read_table(f"{DATA_DIR}/test.parquet").to_pandas()
tcomp = components(test, bank_test)
ranked = rank_keys(tcomp, W)
smi = dict(zip(pool_key, pool.smiles))
rows = [(m, ";".join(smi[k] for k in ranked.get(m, [])[:25]) or "CCO") for m in test.molecule_id.unique()]
sub = pd.DataFrame(rows, columns=["molecule_id", "smiles"]); sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print((sub.smiles == "CCO").sum(), "molecules without candidates"); sub.head()

In [ ]:
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv"); chk = pd.read_csv(f"{OUT_DIR}/submission.csv")
assert list(chk.columns) == ["molecule_id", "smiles"] and chk.molecule_id.is_unique and chk.notna().all().all()
assert set(chk.molecule_id) == set(ss.molecule_id) and chk.smiles.str.count(";").add(1).max() <= 25
print("submission OK", chk.shape)